In [6]:
import pandas as pd
import numpy as np
import re
import pickle
from pathlib import Path
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

BASE_DIR = Path().resolve().parent
DATA_DIR = BASE_DIR / 'data'
FIGURES_DIR = BASE_DIR / 'figures'
MODELS_DIR = BASE_DIR / 'models'

MODELS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

In [5]:
df = pd.read_csv(DATA_DIR / 'enron_spam_data.csv')

print(f"Total de e-mails: {len(df)}")
print(f"\nColunas: {df.columns.tolist()}")
print(f"\nDistribuição de classes:\n{df['Spam/Ham'].value_counts()}")
print(f"\nPrimeiras linhas:\n{df.head(3)}")

Total de e-mails: 33716

Colunas: ['Unnamed: 0', 'Subject', 'Message', 'Spam/Ham', 'Date']

Distribuição de classes:
Spam/Ham
spam    17171
ham     16545
Name: count, dtype: int64

Primeiras linhas:
   Unnamed: 0                       Subject  \
0           0  christmas tree farm pictures   
1           1      vastar resources , inc .   
2           2  calpine daily gas nomination   

                                             Message Spam/Ham        Date  
0                                                NaN      ham  1999-12-10  
1  gary , production from the high island larger ...      ham  1999-12-13  
2             - calpine daily gas nomination 1 . doc      ham  1999-12-14  


In [7]:
df = df.dropna(subset=['Message'])
print(f"Total após remoção de vazios: {len(df)}")

Total após remoção de vazios: 33664


In [8]:
def truncate_to_150_words(text):
    """
    Extrai as 150 primeiras palavras do corpo do e-mail.
    Recorte deliberado adotado como unidade de análise deste trabalho.
    """
    words = str(text).split()
    return ' '.join(words[:150])

df['text_truncated'] = df['Message'].apply(truncate_to_150_words)

# Verificação
df['word_count_truncated'] = df['text_truncated'].str.split().str.len()
print("Distribuição após recorte:")
print(df['word_count_truncated'].describe())
print(f"\nTodos com no máximo 150 palavras: {(df['word_count_truncated'] <= 150).all()}")


Distribuição após recorte:
count    33664.000000
mean       131.455056
std         39.332724
min          1.000000
25%        150.000000
50%        150.000000
75%        150.000000
max        150.000000
Name: word_count_truncated, dtype: float64

Todos com no máximo 150 palavras: True


In [10]:
# Célula 5 — Pré-processamento textual
STOP_WORDS = set(stopwords.words('english'))

def preprocess(text):
    """
    Pipeline de pré-processamento:
    1. Lowercase
    2. Remove caracteres não-alfabéticos
    3. Tokenização
    4. Remoção de stopwords
    """
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 1]
    return ' '.join(tokens)

df['text_processed'] = df['text_truncated'].apply(preprocess)

# Verificação
print("Exemplo de texto original (truncado):")
print(df['text_truncated'].iloc[0])
print("\nApós pré-processamento:")
print(df['text_processed'].iloc[0])

Exemplo de texto original (truncado):
gary , production from the high island larger block a - 1 # 2 commenced on saturday at 2 : 00 p . m . at about 6 , 500 gross . carlos expects between 9 , 500 and 10 , 000 gross for tomorrow . vastar owns 68 % of the gross production . george x 3 - 6992 - - - - - - - - - - - - - - - - - - - - - - forwarded by george weissman / hou / ect on 12 / 13 / 99 10 : 16 am - - - - - - - - - - - - - - - - - - - - - - - - - - - daren j farmer 12 / 10 / 99 10 : 38 am to : carlos j rodriguez / hou / ect @ ect

Após pré-processamento:
gary production high island larger block commenced saturday gross carlos expects gross tomorrow vastar owns gross production george forwarded george weissman hou ect daren farmer carlos rodriguez hou ect ect


In [12]:
before = len(df)
df = df[df['text_processed'].str.strip() != '']
after = len(df)
print(f"Removidos {before - after} e-mails sem conteúdo após pré-processamento.")
print(f"Corpus final: {after} e-mails")
print(f"\nDistribuição final:\n{df['Spam/Ham'].value_counts()}")

Removidos 8 e-mails sem conteúdo após pré-processamento.
Corpus final: 33656 e-mails

Distribuição final:
Spam/Ham
spam    17171
ham     16485
Name: count, dtype: int64


In [13]:
df['label'] = df['Spam/Ham'].map({'spam': 1, 'ham': 0})

print(f"Spam (1): {df['label'].sum()}")
print(f"Ham  (0): {(df['label'] == 0).sum()}")

Spam (1): 17171
Ham  (0): 16485


In [14]:
X = df['text_processed']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Treino : {len(X_train)} e-mails")
print(f"Teste  : {len(X_test)} e-mails")
print(f"\nProporção no treino:\n{y_train.value_counts(normalize=True).round(4)}")
print(f"\nProporção no teste:\n{y_test.value_counts(normalize=True).round(4)}")

Treino : 26924 e-mails
Teste  : 6732 e-mails

Proporção no treino:
label
1    0.5102
0    0.4898
Name: proportion, dtype: float64

Proporção no teste:
label
1    0.5102
0    0.4898
Name: proportion, dtype: float64


In [15]:
tfidf = TfidfVectorizer(
    max_features=10000,   # vocabulário máximo
    sublinear_tf=True,    # aplica log(TF) — melhora desempenho em textos
    min_df=2,             # ignora termos que aparecem em menos de 2 documentos
    ngram_range=(1, 2)    # unigramas e bigramas
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f"Shape treino : {X_train_tfidf.shape}")
print(f"Shape teste  : {X_test_tfidf.shape}")
print(f"\nVocabulário: {len(tfidf.vocabulary_)} termos")
print(f"\nTop 20 termos por TF-IDF médio:")
scores = X_train_tfidf.mean(axis=0).A1
top_idx = scores.argsort()[-20:][::-1]
vocab_inv = {v: k for k, v in tfidf.vocabulary_.items()}
for idx in top_idx:
    print(f"  {vocab_inv[idx]:<25} {scores[idx]:.4f}")

Shape treino : (26924, 10000)
Shape teste  : (6732, 10000)

Vocabulário: 10000 termos

Top 20 termos por TF-IDF médio:
  cost                      0.0286
  ivanhoe                   0.0273
  final                     0.0267
  schedule                  0.0258
  subject                   0.0254
  type                      0.0253
  please                    0.0251
  schedules                 0.0234
  point                     0.0226
  date                      0.0225
  estimate                  0.0221
  perry                     0.0220
  frazier                   0.0220
  mansoor                   0.0220
  abdmoulaie                0.0220
  abdmoulaie mansoor        0.0220
  frazier perry             0.0220
  esd                       0.0220
  get                       0.0219
  enron                     0.0216


In [16]:
objects = {
    'X_train_tfidf': X_train_tfidf,
    'X_test_tfidf':  X_test_tfidf,
    'y_train':       y_train,
    'y_test':        y_test,
    'tfidf':         tfidf,
}

for name, obj in objects.items():
    path = MODELS_DIR / f'{name}.pkl'
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
    print(f"Salvo: {path}")

print("\nPré-processamento concluído.")

Salvo: /home/edson-luiz/Projetos/TCC/models/X_train_tfidf.pkl
Salvo: /home/edson-luiz/Projetos/TCC/models/X_test_tfidf.pkl
Salvo: /home/edson-luiz/Projetos/TCC/models/y_train.pkl
Salvo: /home/edson-luiz/Projetos/TCC/models/y_test.pkl
Salvo: /home/edson-luiz/Projetos/TCC/models/tfidf.pkl

Pré-processamento concluído.
